# Titanic Survival Prediction: A Machine Learning Classification Project

**Objective:** predict whether a passenger survived (`Survived = 1`) or died (`Survived = 0`) in the Titanic disaster, using passenger information such as sex, age, ticket class and fare.

**Dataset:** the Kaggle *Titanic: Machine Learning from Disaster* dataset (`train.csv`, 891 passengers). It is the only file with the `Survived` label, so we use it for both training and testing.

| Column | Meaning |
|---|---|
| `PassengerId` | Row ID |
| `Survived` | **Target**: 0 = No, 1 = Yes |
| `Pclass` | Ticket class (1 = upper, 2 = middle, 3 = lower) |
| `Name` | Passenger name (contains a title such as Mr, Mrs, Miss) |
| `Sex` | male / female |
| `Age` | Age in years |
| `SibSp` | Number of siblings or spouses aboard |
| `Parch` | Number of parents or children aboard |
| `Ticket` | Ticket number |
| `Fare` | Ticket price |
| `Cabin` | Cabin number |
| `Embarked` | Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |

**Project outline**
1. Load and inspect the data
2. Exploratory Data Analysis (univariate, bivariate, multivariate)
3. Feature engineering
4. Train/test split and preprocessing pipeline
5. Train four classification models
6. Evaluation
7. Hyperparameter tuning
8. Model comparison and visual comparison
9. Final conclusion

**Setup:** put Kaggle's `train.csv` in the same folder as this notebook (download from https://www.kaggle.com/c/titanic/data). If the file is missing, the notebook tries to download a public copy automatically.

## 1. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)

sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42
print("Setup complete")

## 2. Load the Dataset

In [ ]:
LOCAL_FILE = "train.csv"
BACKUP_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

if os.path.exists(LOCAL_FILE):
    df = pd.read_csv(LOCAL_FILE)
    print(f"Loaded local file: {LOCAL_FILE}")
else:
    print("train.csv not found, downloading a public copy of the Titanic data...")
    df = pd.read_csv(BACKUP_URL)

print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Summary statistics for numerical columns
df.describe()

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"Missing": missing, "Percent": missing_pct})[missing > 0]

**What we see:**
- `Age` has about 20% missing values, `Cabin` about 77%, and `Embarked` only 2 missing values.
- `Cabin` has too many gaps to impute sensibly, but *whether a cabin was recorded* may still be informative (it hints at wealthier passengers). We will turn it into a flag later.
- The target is moderately imbalanced: more passengers died than survived (checked below).

## 3. Exploratory Data Analysis (EDA)

### 3.1 Univariate Analysis
We look at each variable on its own.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Survived
sns.countplot(data=df, x="Survived", ax=axes[0, 0])
axes[0, 0].set_title("Survived (0 = No, 1 = Yes)")

# Sex
sns.countplot(data=df, x="Sex", ax=axes[0, 1])
axes[0, 1].set_title("Sex")

# Pclass
sns.countplot(data=df, x="Pclass", ax=axes[0, 2])
axes[0, 2].set_title("Passenger Class")

# Age
sns.histplot(df["Age"].dropna(), bins=30, kde=True, ax=axes[1, 0])
axes[1, 0].set_title("Age Distribution")

# Fare
sns.histplot(df["Fare"], bins=40, kde=True, ax=axes[1, 1])
axes[1, 1].set_title("Fare Distribution")

# Embarked
sns.countplot(data=df, x="Embarked", ax=axes[1, 2])
axes[1, 2].set_title("Port of Embarkation")

plt.tight_layout()
plt.show()

print(df["Survived"].value_counts(normalize=True).round(3).rename("Share of passengers"))

**Observations**
- About 38% of passengers survived, so the classes are mildly imbalanced (this is why accuracy alone is not enough, see Section 8).
- There were roughly twice as many men as women.
- Most passengers travelled in 3rd class.
- Age is roughly bell-shaped with a bump of young children. Fare is **heavily right-skewed**: most tickets were cheap and a few were very expensive.
- Most passengers boarded at Southampton (S).

### 3.2 Bivariate Analysis
How does each variable relate to survival?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sex vs Survived
sns.barplot(data=df, x="Sex", y="Survived", ax=axes[0, 0])
axes[0, 0].set_title("Survival Rate by Sex")
axes[0, 0].set_ylabel("Survival rate")

# Pclass vs Survived
sns.barplot(data=df, x="Pclass", y="Survived", ax=axes[0, 1])
axes[0, 1].set_title("Survival Rate by Passenger Class")
axes[0, 1].set_ylabel("Survival rate")

# Age vs Survived
sns.kdeplot(data=df, x="Age", hue="Survived", fill=True, common_norm=False, ax=axes[1, 0])
axes[1, 0].set_title("Age Distribution by Survival")

# Fare vs Survived (log-style view using a box plot with a log scale)
sns.boxplot(data=df, x="Survived", y="Fare", ax=axes[1, 1])
axes[1, 1].set_yscale("log")
axes[1, 1].set_title("Fare by Survival (log scale)")

plt.tight_layout()
plt.show()

**Observations**
- **Sex** is the strongest signal: women survived at roughly 74% versus about 19% for men ("women and children first").
- **Pclass:** survival falls steadily from 1st to 3rd class (about 63% to 47% to 24%).
- **Age:** young children had a higher chance of survival, while many young adults (20-35) did not survive.
- **Fare:** survivors paid higher fares on average, which is consistent with the class effect.

### 3.3 Multivariate Analysis
How do several variables combine?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Survival by Sex and Pclass
sns.barplot(data=df, x="Pclass", y="Survived", hue="Sex", ax=axes[0])
axes[0].set_title("Survival Rate by Pclass and Sex")
axes[0].set_ylabel("Survival rate")

# Survival heatmap
pivot = df.pivot_table(values="Survived", index="Sex", columns="Pclass", aggfunc="mean")
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGnBu", ax=axes[1])
axes[1].set_title("Survival Rate: Sex x Pclass")

plt.tight_layout()
plt.show()

In [ ]:
# Survival based on Age, Sex and Pclass
g = sns.FacetGrid(df.dropna(subset=["Age"]), col="Pclass", row="Sex", hue="Survived",
                  height=3, aspect=1.5, palette={0: "tomato", 1: "seagreen"})
g.map(sns.histplot, "Age", bins=20, alpha=0.6)
g.add_legend(title="Survived")
g.fig.suptitle("Survival by Age, Sex and Pclass", y=1.02)
plt.show()

In [ ]:
# Correlation of numerical features
plt.figure(figsize=(7, 5))
sns.heatmap(df[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]].corr(),
            annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

**Observations**
- Sex and class interact strongly: **almost all 1st and 2nd class women survived**, while **3rd class men had the lowest survival rate**. 3rd class women did much worse than other women (about 50%).
- In the Age/Sex/Pclass grid, boys under about 10 mostly survived in 1st and 2nd class, while adult men in 3rd class mostly did not.
- `Pclass` is negatively correlated with `Survived` and with `Fare` (better class means higher fare, higher survival).

## 4. Feature Engineering

We now create features that capture information the raw columns hide. Each step is explained below.

| Step | Why |
|---|---|
| **Title** from `Name` | Titles (Mr, Mrs, Miss, Master...) encode sex, age group and social status. Rare titles are grouped together to avoid tiny categories. |
| **FamilySize** = `SibSp + Parch + 1` | Family size matters more than the two separate counts. |
| **IsAlone** | Travelling alone had a different survival pattern than travelling with family. |
| **HasCabin** | Having a recorded cabin hints at wealth/class. Raw `Cabin` is 77% missing, so we keep only this flag. |
| **Drop** `PassengerId`, `Name`, `Ticket`, `Cabin`, `SibSp`, `Parch` | IDs and free text carry no general signal; `SibSp`/`Parch` are replaced by `FamilySize`. |

> **Data leakage note:** the steps below use only each passenger's own row (no averages or statistics from other rows), so doing them before the train/test split is safe. Anything that *learns from data* (imputation values, scaling, encoding) is done later inside the pipeline, after the split.

In [ ]:
data = df.copy()

# 1. Title from Name
data["Title"] = data["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
title_map = {
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    "Lady": "Rare", "the Countess": "Rare", "Countess": "Rare", "Sir": "Rare", "Don": "Rare",
    "Dona": "Rare", "Jonkheer": "Rare", "Dr": "Rare", "Rev": "Rare", "Col": "Rare",
    "Major": "Rare", "Capt": "Rare",
}
data["Title"] = data["Title"].replace(title_map)
# Anything still unusual becomes "Rare"
data.loc[~data["Title"].isin(["Mr", "Mrs", "Miss", "Master", "Rare"]), "Title"] = "Rare"

# 2. FamilySize and IsAlone
data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
data["IsAlone"] = (data["FamilySize"] == 1).astype(int)

# 3. HasCabin
data["HasCabin"] = data["Cabin"].notnull().astype(int)

# 4. Treat Pclass as a category (1st/2nd/3rd are labels, not a true numeric scale)
data["Pclass"] = data["Pclass"].astype(str)

# 5. Drop columns we no longer need
data = data.drop(columns=["PassengerId", "Name", "Ticket", "Cabin", "SibSp", "Parch"])

data.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.barplot(data=data, x="Title", y="Survived", ax=axes[0]); axes[0].set_title("Survival Rate by Title")
sns.barplot(data=data, x="FamilySize", y="Survived", ax=axes[1]); axes[1].set_title("Survival Rate by FamilySize")
sns.barplot(data=data, x="HasCabin", y="Survived", ax=axes[2]); axes[2].set_title("Survival Rate by HasCabin")
plt.tight_layout()
plt.show()

**Observations:** the new features are informative. `Mrs` and `Miss` have high survival, `Mr` very low, and `Master` (young boys) is high. Small families (2-4 people) survived more than singles or very large families. Passengers with a recorded cabin survived much more often.

Missing values (`Age`, `Embarked`) and encoding/scaling are handled next, **inside the pipeline**, so that they are learned only from training data.

## 5. Train/Test Split and Preprocessing Pipeline

In [ ]:
X = data.drop(columns="Survived")
y = data["Survived"]

# Stratify keeps the same survived/died ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Training set:", X_train.shape, "| Test set:", X_test.shape)
print("Survival rate (train):", round(y_train.mean(), 3), "| (test):", round(y_test.mean(), 3))

### Preprocessing pipeline
| Feature type | Columns | Steps |
|---|---|---|
| Numerical | `Age`, `Fare`, `FamilySize` | Median imputation, then `StandardScaler` |
| Categorical | `Pclass`, `Sex`, `Embarked`, `Title` | Most-frequent imputation, then `OneHotEncoder` |
| Binary (0/1) | `IsAlone`, `HasCabin` | Passed through unchanged |

The `ColumnTransformer` is placed inside a `Pipeline` together with the model. When we call `.fit()` on the training data, the imputer medians, scaler means and encoder categories are learned **only from the training set**, then applied to the test set. This prevents data leakage. Scaling matters for Logistic Regression and SVM (distance/gradient based) and is harmless for tree models, so we use the same preprocessing for all four models.

In [ ]:
numeric_features = ["Age", "Fare", "FamilySize"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title"]
binary_features = ["IsAlone", "HasCabin"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
    ("bin", "passthrough", binary_features),
])
preprocessor

## 6. Train the Models
We train four classifiers, each inside the **same** preprocessing pipeline.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "SVM": SVC(probability=True, random_state=RANDOM_STATE),  # probability=True enables ROC-AUC
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

def make_pipeline(model):
    return Pipeline([("preprocess", preprocessor), ("model", model)])

base_pipelines = {}
for name, model in models.items():
    pipe = make_pipeline(model)
    pipe.fit(X_train, y_train)
    base_pipelines[name] = pipe
    print(f"Trained: {name}")

## 7. Model Evaluation

### Why accuracy alone is not enough
Accuracy is the share of correct predictions. Here about 62% of passengers did not survive, so a useless model that always predicts "died" would already score ~62%. Accuracy also treats all mistakes equally. We therefore also use:

- **Precision**: of the passengers we predicted would survive, how many really did? (Avoids false alarms.)
- **Recall**: of the passengers who really survived, how many did we find? (Avoids missing survivors.)
- **F1 score**: the harmonic mean of precision and recall, a single balanced score that is robust to class imbalance. **We use it as our main metric.**
- **ROC-AUC**: measures how well the model *ranks* survivors above non-survivors across all thresholds, independent of the cut-off chosen.
- **Confusion matrix**: shows exactly which kinds of errors are made (false positives vs false negatives).

In [ ]:
def evaluate(pipe, X_te, y_te):
    """Return a dict of metrics for a fitted pipeline."""
    y_pred = pipe.predict(X_te)
    y_prob = pipe.predict_proba(X_te)[:, 1]
    return {
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred),
        "Recall": recall_score(y_te, y_pred),
        "F1 Score": f1_score(y_te, y_pred),
        "ROC-AUC": roc_auc_score(y_te, y_prob),
    }

base_results = pd.DataFrame({name: evaluate(p, X_test, y_test) for name, p in base_pipelines.items()}).T
base_results.index.name = "Model"
print("Baseline (default parameters) performance on the test set")
base_results.round(3)

In [ ]:
def show_confusion_matrices(pipes, title):
    fig, axes = plt.subplots(1, len(pipes), figsize=(5 * len(pipes), 4.5))
    for ax, (name, pipe) in zip(axes, pipes.items()):
        ConfusionMatrixDisplay.from_estimator(
            pipe, X_test, y_test, display_labels=["Died", "Survived"],
            cmap="Blues", colorbar=False, ax=ax
        )
        ax.set_title(name)
        ax.grid(False)
    fig.suptitle(title, y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

show_confusion_matrices(base_pipelines, "Confusion Matrices (Baseline Models)")

In [ ]:
for name, pipe in base_pipelines.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    print(classification_report(y_test, pipe.predict(X_test), target_names=["Died", "Survived"]))

**Reading the results:** in the confusion matrices, the diagonal shows correct predictions. *Died, predicted Survived* are false positives; *Survived, predicted Died* are false negatives. Models usually find non-survivors (the majority class) more easily than survivors, so Recall for the "Survived" class is the weaker number to watch. With only 179 test passengers, small differences between models can be noise, so we also use cross-validation during tuning.

## 8. Hyperparameter Tuning

We use `GridSearchCV` with **5-fold stratified cross-validation** on the **training set only**. The test set is never used to choose parameters, so it stays an honest final check. We optimise **F1** because it balances precision and recall. In a `Pipeline`, parameters are addressed as `model__<parameter>`.

In [ ]:
param_grids = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__penalty": ["l2"],
    },
    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 5, 8],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
    },
    "SVM": {
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", 0.01, 0.1],
        "model__kernel": ["rbf", "linear"],
    },
    "Gradient Boosting": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [2, 3, 4],
        "model__subsample": [0.8, 1.0],
    },
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
tuned_pipelines = {}
best_params = {}

for name, model in models.items():
    print(f"Tuning {name} ...")
    grid = GridSearchCV(
        make_pipeline(model), param_grids[name],
        cv=cv, scoring="f1", n_jobs=-1
    )
    grid.fit(X_train, y_train)
    tuned_pipelines[name] = grid.best_estimator_
    best_params[name] = grid.best_params_
    print(f"   Best CV F1: {grid.best_score_:.3f} | Best params: {grid.best_params_}")

### Evaluate the tuned models on the test set

In [ ]:
tuned_results = pd.DataFrame({name: evaluate(p, X_test, y_test) for name, p in tuned_pipelines.items()}).T
tuned_results.index.name = "Model"
tuned_results.round(3)

In [ ]:
show_confusion_matrices(tuned_pipelines, "Confusion Matrices (Tuned Models)")

In [ ]:
for name, pipe in tuned_pipelines.items():
    print("=" * 60)
    print(name, "(tuned)")
    print("=" * 60)
    print(classification_report(y_test, pipe.predict(X_test), target_names=["Died", "Survived"]))

## 9. Model Comparison

In [ ]:
comparison = tuned_results.round(3).reset_index()
comparison

In [ ]:
# Baseline vs tuned F1 (shows whether tuning helped)
compare_tuning = pd.DataFrame({
    "F1 (default)": base_results["F1 Score"],
    "F1 (tuned)": tuned_results["F1 Score"],
}).round(3)
compare_tuning["Change"] = (compare_tuning["F1 (tuned)"] - compare_tuning["F1 (default)"]).round(3)
compare_tuning

In [ ]:
best_accuracy = tuned_results["Accuracy"].idxmax()
best_recall = tuned_results["Recall"].idxmax()
best_f1 = tuned_results["F1 Score"].idxmax()
best_auc = tuned_results["ROC-AUC"].idxmax()

# Recommended model: highest F1, using ROC-AUC to break ties
recommended = tuned_results.sort_values(["F1 Score", "ROC-AUC"], ascending=False).index[0]

print(f"Best Accuracy : {best_accuracy} ({tuned_results.loc[best_accuracy, 'Accuracy']:.3f})")
print(f"Best Recall   : {best_recall} ({tuned_results.loc[best_recall, 'Recall']:.3f})")
print(f"Best F1 Score : {best_f1} ({tuned_results.loc[best_f1, 'F1 Score']:.3f})")
print(f"Best ROC-AUC  : {best_auc} ({tuned_results.loc[best_auc, 'ROC-AUC']:.3f})")
print()
print(f"Overall recommended model: {recommended}")

## 10. Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ["Accuracy", "F1 Score"]):
    sns.barplot(x=tuned_results.index, y=tuned_results[metric], ax=ax)
    ax.set_title(f"{metric} by Model (Tuned)")
    ax.set_ylim(0.5, 1.0)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f")

plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart of all metrics
tuned_results.plot(kind="bar", figsize=(12, 5), width=0.8)
plt.title("All Metrics by Model (Tuned)")
plt.ylabel("Score")
plt.ylim(0.5, 1.0)
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.show()

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(7, 6))
for name, pipe in tuned_pipelines.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random guess")
ax.set_title("ROC Curves (Tuned Models)")
ax.legend(loc="lower right")
plt.show()

## 11. Which Features Were Important?

In [ ]:
# Use the tuned Random Forest to see which features matter most
rf_pipe = tuned_pipelines["Random Forest"]
feature_names = rf_pipe.named_steps["preprocess"].get_feature_names_out()
feature_names = [n.split("__", 1)[1] for n in feature_names]   # tidy names
importances = pd.Series(rf_pipe.named_steps["model"].feature_importances_, index=feature_names)

plt.figure(figsize=(8, 6))
importances.sort_values().tail(12).plot(kind="barh")
plt.title("Top Feature Importances (Tuned Random Forest)")
plt.xlabel("Importance")
plt.show()

importances.sort_values(ascending=False).head(8).round(3)

In [ ]:
# Logistic Regression coefficients give direction (positive = more likely to survive)
lr_pipe = tuned_pipelines["Logistic Regression"]
coefs = pd.Series(lr_pipe.named_steps["model"].coef_[0], index=feature_names)

plt.figure(figsize=(8, 6))
coefs.sort_values().plot(kind="barh", color=(coefs.sort_values() > 0).map({True: "seagreen", False: "tomato"}))
plt.title("Logistic Regression Coefficients (Tuned)")
plt.xlabel("Coefficient (green = increases survival odds)")
plt.show()

## 12. Final Conclusion

In [ ]:
print("FINAL SUMMARY")
print("-" * 60)
print(tuned_results.round(3).to_string())
print()
print(f"Best model by F1 : {best_f1}")
print(f"Best by Recall   : {best_recall}")
print(f"Recommended      : {recommended}")
print("Top 5 features   :", ", ".join(importances.sort_values(ascending=False).head(5).index))

### 1. What was learned from the dataset
Survival on the Titanic was strongly shaped by **sex, class and age**. Women, children and first-class passengers had much higher survival rates than men and third-class passengers, reflecting the "women and children first" policy and the position of cabins relative to the lifeboats. `Age` and `Cabin` had many missing values and `Fare` was heavily skewed, so careful preprocessing was necessary.

### 2. Which features were important
The feature-importance and coefficient plots above show which inputs matter most. Typically **Sex (and the Title `Mr`)**, **Fare**, **Age**, **Pclass** and **FamilySize** dominate. The engineered `Title` feature is especially useful because it combines sex, age group and social status in one variable, and `FamilySize`/`HasCabin` add information the raw columns did not expose directly.

### 3. Which model performed best
See the comparison table and the printed summary in Sections 9 and 12 for the exact numbers. The model with the highest tuned F1 score (with ROC-AUC as tie-breaker) is reported as the *recommended* model, and the best-Recall and best-Accuracy models are listed separately. On this dataset the four models usually land close together (roughly 80-84% accuracy), so differences are small and, with only 179 test passengers, partly due to chance.

### 4. Why that model was selected
We selected the model with the best balance of precision and recall (F1), supported by a high ROC-AUC, rather than the one with the best accuracy alone. Hyperparameters were chosen by 5-fold cross-validation on the training set only, and the same preprocessing pipeline was used for all models, so the comparison is fair and free from data leakage.

### 5. Possible improvements
- Extract more from the data: cabin deck letter, ticket prefix/group size, surname-based family groups.
- Impute `Age` more intelligently (e.g. median age per Title).
- Use repeated or nested cross-validation for more reliable model comparison on such a small dataset.
- Try `RandomizedSearchCV` with wider ranges, or XGBoost / LightGBM.
- Tune the decision threshold if recall (finding survivors) matters more than precision.
- Use SHAP values for more detailed model explanations.
- Submit predictions on Kaggle's `test.csv` to measure performance on truly unseen data.